In [1]:
!pip install -r requirements.txt

In [2]:
from orchestrator_agent import MultiAgent
import os
from pathlib import Path
from test_cases import TEST_CASES
import json
os.environ["OPENAI_API_KEY"] = Path("open_ai_api_key.txt").read_text().strip()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
agent = MultiAgent()


In [4]:
result_traces = []
for case in TEST_CASES: 
    answer,_,trace_info = await agent.answer(case["prompt"])
    # Add the response to the traces
    trace_info["answer"] = answer
    #Append to result traces
    result_traces.append(trace_info)
    print(f"Case {case["id"]} Done")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Case 1 Done
Case 2 Done
Case 3 Done
Case 4 Done
Case 5 Done
Case 6 Done
Case 7 Done
Case 8 Done
Case 9 Done
Case 10 Done
Case 11 Done
Case 12 Done
Case 13 Done
Case 14 Done
Case 15 Done
Case 16 Done
Case 17 Done
Case 18 Done
Case 19 Done
Case 20 Done
Case 21 Done


In [5]:

#dump to file 
json.dump(result_traces, open("eval_suite.json", "w"), indent=2, default=str)
#print(result_traces)

In [7]:
import pandas as pd

with open("eval_suite.json","r", encoding="utf-8") as file:
    data = json.load(file)

for i in range(len(data)):
    # Stamp the category of the test case
    data[i]["category"] = TEST_CASES[i]["name"]
    # Stamp the response notes from the test case
    data[i]["expected_response_notes"] = TEST_CASES[i]["expectations"]["response_notes"]
    
    # Check if sources expectation matches
    data[i]["rag_source_pass"] = (not (data[i]["sources"]) == (not TEST_CASES[i]["expectations"]["sources_rag"]) )

    # Check guardrails
    guard = data[i].get("guardrail_tripped", False)
    data[i]["guardrail_pass"] = (guard == TEST_CASES[i]["expectations"]["guardrail_trip"])

    # Check expected tools 
    data[i]["tool_pass"]= False
    if TEST_CASES[i]["expectations"]["expected_tools"]:
        for entry in TEST_CASES[i]["expectations"]["expected_tools"]:
            if entry in data[i]["tools_called"]:
                data[i]["tool_pass"]=True
    else:
        data[i]["tool_pass"]=(TEST_CASES[i]["expectations"]["expected_tools"]==data[i]["tools_called"])
 
    
    # Check sources if relevant
    if TEST_CASES[i]["expectations"]["notice_ids"]:
        #Default is fail
        data[i]["correct_notices"] = False
        for entry in data[i]["sources"]:
            if entry["notice_id"] in TEST_CASES[i]["expectations"]["notice_ids"]:
                data[i]["correct_notices"] = True
    else:
        data[i]["correct_notices"] = "N/A"
        

df = pd.DataFrame(data)   

average_latency = df["latency_ms"].mean()
print(f"Average Latency: {average_latency} ms")
    

Average Latency: 14907.10476190476 ms


In [8]:
df

,question,tool_calls,tools_called,sources,latency_ms,answer,category,expected_response_notes,rag_source_pass,guardrail_pass,tool_pass,correct_notices,guardrail_tripped,guardrail_reason
0,Tell me about the public meeting on beavers,"[{'name': 'rag', 'input': {'input': 'public me...",[rag],[],12512.9,There currently isn't a public meeting on beav...,Non-existant meeting,Should not find any matching public notices,True,True,True,N/A,NaN,NaN
1,Tell me about the city meeting on unicorns sig...,[],[],[],1426.6,"Sorry, that doesn't seem to be related to the ...",Non-existant meeting,Unicorns trip the guardrail,True,True,True,N/A,True,The mention of a city meeting suggests a conne...
2,Can I testify at the August 6th Tree Removal H...,"[{'name': 'rag', 'input': {'input': 'August 6t...",[rag],"[{'notice_id': '16602326', 'title': 'Tree Remo...",7684.9,"No, you cannot testify at the Tree Removal Hea...",Public testimony,Valid meeting but no public testimony at this ...,True,True,True,True,NaN,NaN
3,Can I testify at the August 11th Zoning Board ...,"[{'name': 'rag', 'input': {'input': 'August 11...",[rag],"[{'notice_id': '16602821', 'title': 'Zoning Bo...",7470.7,"Yes, you can testify at the Zoning Board of Ap...",Public testimony,Valid meeting and public testimony allowed at ...,True,True,True,True,NaN,NaN
4,Is the Boston Landmarks Commission meeting hap...,"[{'name': 'rag', 'input': {'input': 'Boston La...",[rag],"[{'notice_id': '16603691', 'title': 'City Coun...",8537.6,The Boston Landmarks Commission meeting schedu...,Cancelled meeting,Valid meeting but was cancelled,True,True,True,True,NaN,NaN
5,Is the August 19th St Botolph area meeting hap...,"[{'name': 'rag', 'input': {'input': 'August 19...",[rag],"[{'notice_id': '16603531', 'title': 'St.Botolp...",8357.3,"No, the St. Botolph Area Architectural Conserv...",Cancelled meeting,Valid meeting but was cancelled,True,True,True,True,NaN,NaN
6,What is Docket #0218 from the City Council Com...,"[{'name': 'rag', 'input': {'input': 'Docket #0...",[rag],"[{'notice_id': '16600016', 'title': 'City Coun...",8397.7,Docket #0218 from the City Council Committee o...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16600016,True,True,True,True,NaN,NaN
7,What is Docket #0932 from the City Council Com...,"[{'name': 'rag', 'input': {'input': 'Docket #0...",[rag],"[{'notice_id': '16594651', 'title': 'City Coun...",8669.3,Docket #0932 from the City Council Committee o...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16600081,True,True,True,True,NaN,NaN
8,Explain crypto wallets,[],[],[],1290.4,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,True,N/A,True,"The query is about crypto wallets, which is a ..."
9,Write a fun limeric,[],[],[],2755.2,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,True,N/A,True,"The query requests a limerick, which is relate..."


In [9]:


print(data[19]["answer"])

print(data[20]["answer"])

In July 2026, there is a scheduled City Council Meeting on July 8, 2026, at 12:00 PM EDT. The agenda for this meeting includes reports from committees and public officers; however, public testimony will not be permitted.
The next City Council Meeting is scheduled for September 23, 2026, at 12:00 PM EDT.
The most recent City Council Meeting took place on September 16, 2026, at 12:00 PM EDT. It's worth noting that public testimony was not permitted during this meeting.


In [10]:
# Investigating differences in latency
only_rag = df[df["tools_called"].apply(lambda x: x ==["rag"])]["latency_ms"]
web_and_rag = df[df["tools_called"].apply(lambda x: x ==["rag","web_search"])]["latency_ms"]
no_tools = df[df["tools_called"].apply(lambda x: x ==[])]["latency_ms"]

print(f"Latency when only RAG: {only_rag.mean()} ms")
print(f"Latency when RAG and Web Search: {web_and_rag.mean()} ms")
print(f"Latency when no tools called (guardrails): {no_tools.mean()} ms")

Latency when only RAG: 8665.109090909093 ms
Latency when RAG and Web Search: 31728.333333333332 ms
Latency when no tools called (guardrails): 1570.2 ms
